# Régressions du salaire selon le sexe, l’âge, la CSP et le secteur

On étudie les associations entre le salaire net moyen mensuel, le sexe, la tranche d’âge, la catégorie socio-professionnelle (CSP) et le secteur d’activité. L’analyse porte sur les données du secteur privé, en euros constants.

Les modèles incluent des effets fixes par année lorsque la comparaison porte sur plusieurs groupes, et regroupent les erreurs-types par année. Après les analyses par âge et par CSP, les sections 9 à 11 étudient successivement l’évolution de l’écart femmes-hommes dans le temps, ses variations selon la CSP, puis selon les quantiles de salaire. Les observations sont des **moyennes ou quantiles agrégés par groupe** : les résultats décrivent des associations, pas des effets causaux individuels.

## 1. Chargement et préparation des données

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf


def formater_p_value(valeur):
    if valeur < 0.001:
        return '< 0,001'
    return f'{valeur:.3f}'.replace('.', ',')


def test_conjoint(resultat, termes_testes):
    if not termes_testes:
        raise ValueError('La liste des coefficients à tester ne peut pas être vide.')
    matrice = np.zeros((len(termes_testes), len(resultat.params)))
    for ligne, terme in enumerate(termes_testes):
        matrice[ligne, resultat.params.index.get_loc(terme)] = 1
    return resultat.wald_test(matrice, use_f=True, scalar=True)

def trouver_racine_projet():
    courant = Path.cwd().resolve()
    for dossier in (courant, *courant.parents):
        if (dossier / 'Data nettoyees').is_dir():
            return dossier
    raise FileNotFoundError(
        'Dossier « Data nettoyees » introuvable. '
        'Lancez ce notebook depuis le projet ou un de ses sous-dossiers.'
    )


RACINE = trouver_racine_projet()
CHEMIN_DONNEES = (
    RACINE
    / 'Data nettoyees'
    / 'DS_DERA_PRIVE_SERIES_LONGUES_data_nettoye.csv'
)
if not CHEMIN_DONNEES.is_file():
    raise FileNotFoundError(
        f'Fichier de données nettoyées absent : {CHEMIN_DONNEES}. '
        'Lancez d’abord Nettoyage_donnees.ipynb.'
    )

prive = pd.read_csv(CHEMIN_DONNEES, sep=';', encoding='utf-8-sig')
print(f'Fichier chargé : {CHEMIN_DONNEES.name} ({len(prive):,} lignes)')
print(f'Colonnes disponibles : {", ".join(prive.columns)}')

Fichier chargé : DS_DERA_PRIVE_SERIES_LONGUES_data_nettoye.csv (14,768 lignes)
Colonnes disponibles : geo, zone_geo, frequence, secteur_activite, tranche_age, taille_entreprise, categorie_socio_prof, sexe, temps_travail, centile_salaire, indicateur, unite, annee, valeur


## 2. Construction de l’échantillon

On conserve le salaire net moyen mensuel, pour la France et l’ensemble des secteurs, âges, tailles d’entreprise et temps de travail. On écarte les modalités « Ensemble » du sexe et de la CSP afin de comparer les femmes et les hommes dans chacune des CSP.

In [2]:
SALAIRE = 'Salaire net moyen mensuel (Euros constants)'
colonnes_requises = {
    'indicateur', 'centile_salaire', 'geo', 'secteur_activite',
    'tranche_age', 'taille_entreprise', 'temps_travail',
    'categorie_socio_prof', 'sexe', 'annee', 'valeur',
}
colonnes_absentes = colonnes_requises.difference(prive.columns)
if colonnes_absentes:
    raise ValueError(f'Colonnes manquantes dans le fichier : {sorted(colonnes_absentes)}')

echantillon = prive.loc[
    prive['indicateur'].eq(SALAIRE)
    & prive['centile_salaire'].eq('Ensemble')
    & prive['geo'].eq('F')
    & prive['secteur_activite'].eq('Ensemble des secteurs')
    & prive['tranche_age'].eq('Ensemble')
    & prive['taille_entreprise'].eq('Ensemble')
    & prive['temps_travail'].eq('Ensemble')
    & prive['sexe'].isin(['Femme', 'Homme'])
    & prive['categorie_socio_prof'].ne('Ensemble'),
    ['annee', 'sexe', 'categorie_socio_prof', 'valeur'],
].copy()

echantillon['annee'] = pd.to_numeric(echantillon['annee'], errors='coerce')
echantillon['valeur'] = pd.to_numeric(echantillon['valeur'], errors='coerce')
if echantillon[['annee', 'valeur']].isna().any().any():
    raise ValueError('Des années ou des salaires sont manquants ou non numériques dans l’échantillon.')
echantillon['annee'] = echantillon['annee'].astype(int)
echantillon['femme'] = echantillon['sexe'].eq('Femme').astype(int)

cles = ['annee', 'sexe', 'categorie_socio_prof']
if echantillon.duplicated(cles).any():
    raise ValueError(f'Plusieurs observations pour les mêmes clés : {cles}')

annees = sorted(echantillon['annee'].unique())
sexes = ['Femme', 'Homme']
csp = sorted(echantillon['categorie_socio_prof'].unique())
attendues = pd.MultiIndex.from_product(
    [annees, sexes, csp], names=cles
)
observees = pd.MultiIndex.from_frame(echantillon[cles])
manquantes = attendues.difference(observees)
if len(manquantes):
    raise ValueError(f'Panel incomplet : {len(manquantes)} combinaisons année-sexe-CSP manquent.')

print(
    f"Échantillon : {len(echantillon)} observations, {len(annees)} années "
    f"({min(annees)}–{max(annees)}), {len(csp)} CSP et 2 sexes."
)
print(f'CSP observées : {", ".join(csp)}')
display(echantillon.head())

Échantillon : 232 observations, 29 années (1996–2024), 4 CSP et 2 sexes.
CSP observées : Cadres et prof. intellectuelles sup., Employés, Ouvriers, Professions intermédiaires


,annee,sexe,categorie_socio_prof,valeur,femme
14,2009,Femme,Professions intermédiaires,2491.0,1
84,2014,Femme,Professions intermédiaires,2490.0,1
87,2021,Femme,Ouvriers,1843.0,1
322,2016,Femme,Professions intermédiaires,2510.0,1
445,2012,Homme,Professions intermédiaires,2855.0,0


## 9. Régression n° 1 – L’évolution de l’écart salarial dans le temps

La spécification compare les salaires des femmes et des hommes année par année :

$$\ln(Salaire_{s,t}) = \alpha + \beta_1 Femme_s + \beta_2 Trend_t + \beta_3(Femme_s \times Trend_t) + \varepsilon_{s,t}$$

`Trend` vaut 0 en 1996, 1 en 1997, etc. Le coefficient central est $\beta_3$ : positif, il indique que le rapport salarial femmes-hommes augmente au fil du temps ; négatif, qu’il diminue. Il s’agit d’une tendance différentielle, pas de l’effet causal d’une politique.

**Limite de la source disponible :** le fichier du projet fournit le salaire net moyen **mensuel** en euros constants, et non le salaire annuel moyen EQTP du tableau EQTP02. La régression utilise donc cette série mensuelle (l’interprétation du coefficient en logarithme reste en variation relative). Le panel retenu couvre 1996–2024. L’écart annuel logarithmique permet d’estimer la pente différentielle avec une erreur-type Newey–West, robuste à l’autocorrélation d’ordre 1.

In [ ]:
echantillon_tendance = prive.loc[
    prive['indicateur'].eq(SALAIRE)
    & prive['centile_salaire'].eq('Ensemble')
    & prive['geo'].eq('F')
    & prive['secteur_activite'].eq('Ensemble des secteurs')
    & prive['tranche_age'].eq('Ensemble')
    & prive['taille_entreprise'].eq('Ensemble')
    & prive['temps_travail'].eq('Ensemble')
    & prive['categorie_socio_prof'].eq('Ensemble')
    & prive['sexe'].isin(['Femme', 'Homme']),
    ['annee', 'sexe', 'valeur'],
].copy()
echantillon_tendance['annee'] = pd.to_numeric(
    echantillon_tendance['annee'], errors='coerce'
)
echantillon_tendance['valeur'] = pd.to_numeric(
    echantillon_tendance['valeur'], errors='coerce'
)
if echantillon_tendance[['annee', 'valeur']].isna().any().any():
    raise ValueError('Des années ou des salaires manquent dans la série temporelle.')
echantillon_tendance['annee'] = echantillon_tendance['annee'].astype(int)
if (echantillon_tendance['valeur'] <= 0).any():
    raise ValueError('Le logarithme du salaire exige des valeurs strictement positives.')

cles_tendance = ['annee', 'sexe']
if echantillon_tendance.duplicated(cles_tendance).any():
    raise ValueError(f'Plusieurs observations pour les mêmes clés : {cles_tendance}')
salaire_par_sexe = echantillon_tendance.pivot(
    index='annee', columns='sexe', values='valeur'
).sort_index()
if salaire_par_sexe[['Femme', 'Homme']].isna().any().any():
    raise ValueError('La série temporelle ne contient pas les deux sexes pour chaque année.')
if 1996 not in salaire_par_sexe.index:
    raise ValueError('La série ne contient pas l’année de référence 1996.')

echantillon_tendance['femme'] = echantillon_tendance['sexe'].eq('Femme').astype(int)
echantillon_tendance['trend'] = echantillon_tendance['annee'] - 1996
modele_specification_tendance = smf.ols(
    'np.log(valeur) ~ femme * trend', data=echantillon_tendance
).fit()

ecarts_annuels = pd.DataFrame({
    'ecart_log': np.log(salaire_par_sexe['Femme'])
    - np.log(salaire_par_sexe['Homme']),
})
ecarts_annuels['trend'] = ecarts_annuels.index - 1996
modele_tendance = smf.ols(
    'ecart_log ~ trend', data=ecarts_annuels
).fit(
    cov_type='HAC',
    cov_kwds={'maxlags': 1, 'use_correction': True},
)
if not np.isclose(
    modele_specification_tendance.params['femme:trend'],
    modele_tendance.params['trend'],
):
    raise ValueError('La pente de l’écart annuel ne correspond pas à l’interaction Femme × Trend.')

terme_tendance = 'trend'
intervalle_tendance = modele_tendance.conf_int().loc[terme_tendance]
beta3 = modele_tendance.params[terme_tendance]
table_tendance = pd.DataFrame([{
    'β₃ : Femme × Trend (log/an)': beta3,
    'IC 95 % bas': intervalle_tendance.iloc[0],
    'IC 95 % haut': intervalle_tendance.iloc[1],
    'Variation relative annuelle (%)': 100 * np.expm1(beta3),
    'p-value': modele_tendance.pvalues[terme_tendance],
}])
table_tendance_affichage = table_tendance.copy()
table_tendance_affichage['p-value'] = (
    table_tendance_affichage['p-value'].map(formater_p_value)
)
display(table_tendance_affichage.round(4))
print(
    f"Observations : {int(modele_specification_tendance.nobs)} ; "
    f"années : {len(ecarts_annuels)} ({min(ecarts_annuels.index)}–"
    f"{max(ecarts_annuels.index)}) ; erreurs-types Newey–West, 1 retard."
)
# β₃ décrit la variation annuelle du rapport salarial femmes-hommes.
pente_pct = 100 * np.expm1(beta3)
pente_ic_pct = 100 * np.expm1(intervalle_tendance.to_numpy())
pente_p = modele_tendance.pvalues[terme_tendance]
beta0 = modele_tendance.params['Intercept']
beta0_ic = modele_tendance.conf_int().loc['Intercept'].to_numpy()
ecarts_1996_pct = 100 * np.expm1(beta0)
ecarts_1996_ic_pct = 100 * np.expm1(beta0_ic)
nb_annees = int(ecarts_annuels.index.max() - ecarts_annuels.index.min())
variation_cumulee_pct = 100 * np.expm1(beta3 * nb_annees)

print(
    f"En 1996, le salaire moyen des femmes représente {100 + ecarts_1996_pct:.1f} % "
    f"de celui des hommes (IC 95 % : {ecarts_1996_ic_pct[0]:.1f} % à "
    f"{ecarts_1996_ic_pct[1]:.1f} %)."
)
print(
    f"La pente β₃ correspond à une évolution de {pente_pct:+.2f} % par an "
    f"du rapport salarial femmes-hommes (IC 95 % : {pente_ic_pct[0]:+.2f} % à "
    f"{pente_ic_pct[1]:+.2f} % ; p = {formater_p_value(pente_p)})."
)
if pente_p < 0.05:
    direction = 's’améliore' if beta3 > 0 else 'se détériore'
    print(
        f"La tendance est statistiquement discernable au seuil de 5 % : le rapport "
        f"salarial femmes-hommes {direction} en moyenne. Sur {nb_annees} ans, "
        f"la pente correspond à une variation relative cumulée d’environ "
        f"{variation_cumulee_pct:+.1f} % du rapport, selon la tendance linéaire estimée."
    )
else:
    print(
        "La pente n’est pas statistiquement discernable au seuil de 5 % : "
        "les données ne permettent pas de conclure à une amélioration ou à une "
        "détérioration régulière du rapport salarial sur la période."
    )
print(
    "Cette pente décrit une évolution moyenne de salaires agrégés ; elle ne "
    "signifie pas que l’écart s’est réduit chaque année, ni qu’un facteur précis "
    "a causé cette évolution."
)

sensibilite_tendance = []
for debut in [1996, 2000, 2010]:
    sous_periode = ecarts_annuels.loc[ecarts_annuels.index >= debut].copy()
    resultat_periode = smf.ols(
        'ecart_log ~ trend', data=sous_periode
    ).fit(
        cov_type='HAC',
        cov_kwds={'maxlags': 1, 'use_correction': True},
    )
    intervalle = resultat_periode.conf_int().loc['trend']
    sensibilite_tendance.append({
        'Période': f'{debut}–{sous_periode.index.max()}',
        'β₃ (log/an)': resultat_periode.params['trend'],
        'IC 95 % bas': intervalle.iloc[0],
        'IC 95 % haut': intervalle.iloc[1],
        'p-value': resultat_periode.pvalues['trend'],
    })
table_sensibilite_tendance = pd.DataFrame(sensibilite_tendance).set_index('Période')
table_sensibilite_tendance_affichage = table_sensibilite_tendance.copy()
table_sensibilite_tendance_affichage['p-value'] = (
    table_sensibilite_tendance_affichage['p-value'].map(formater_p_value)
)
display(table_sensibilite_tendance_affichage.round(4))

## 10. Régression n° 2 – Les écarts salariaux selon la CSP

Le modèle utilise les salaires nets moyens mensuels par sexe et CSP, en logarithme, avec effets fixes d’année. Les ouvriers et les hommes sont les catégories de référence :

$$\ln(Salaire_{s,c,t}) = \alpha + \beta_1 Femme_s + \beta_2 CSP_c + \beta_3(Femme_s \times CSP_c) + \lambda_t + \varepsilon_{s,c,t}$$

Les interactions mesurent si l’écart femmes-hommes diffère selon la CSP. Les erreurs-types sont regroupées par année. En complément, une seconde spécification ajoute `Femme × CSP × Trend` pour mesurer si l’évolution de l’écart varie selon la CSP. Les effets fixes absorbent la tendance commune : les tendances CSP sont donc exprimées relativement aux ouvriers, et le terme de tendance commun n’est pas estimé séparément. Cette extension reste descriptive.

In [ ]:
echantillon_csp = echantillon.copy()
if (echantillon_csp['valeur'] <= 0).any():
    raise ValueError('Le logarithme du salaire exige des valeurs strictement positives.')

formule_csp = (
    "np.log(valeur) ~ femme * "
    "C(categorie_socio_prof, Treatment(reference='Ouvriers')) + C(annee)"
)
modele_csp = smf.ols(
    formula=formule_csp, data=echantillon_csp
).fit(
    cov_type='cluster',
    cov_kwds={'groups': echantillon_csp['annee'], 'use_correction': True},
)
if modele_csp.model.rank < len(modele_csp.params):
    raise ValueError('Le modèle sexe-CSP n’est pas de rang complet.')

termes_interactions_csp = [
    terme for terme in modele_csp.params.index
    if terme.startswith(
        "femme:C(categorie_socio_prof, Treatment(reference='Ouvriers'))"
    )
]
if len(termes_interactions_csp) != echantillon_csp['categorie_socio_prof'].nunique() - 1:
    raise ValueError('Les interactions sexe-CSP attendues ne sont pas toutes estimées.')

ecarts_csp = []
for categorie in ['Ouvriers', *sorted(
    c for c in echantillon_csp['categorie_socio_prof'].unique()
    if c != 'Ouvriers'
)]:
    contraste = np.zeros(len(modele_csp.params))
    contraste[modele_csp.params.index.get_loc('femme')] = 1
    if categorie != 'Ouvriers':
        terme = (
            "femme:C(categorie_socio_prof, Treatment(reference='Ouvriers'))"
            f"[T.{categorie}]"
        )
        if terme not in modele_csp.params.index:
            raise KeyError(f'Interaction sexe-CSP introuvable pour : {categorie}')
        contraste[modele_csp.params.index.get_loc(terme)] = 1
    test_ecart = modele_csp.t_test(contraste)
    effet = float(np.asarray(test_ecart.effect).reshape(-1)[0])
    intervalle = np.asarray(test_ecart.conf_int()).reshape(-1)
    ecarts_csp.append({
        'CSP': categorie,
        'Écart Femme–Homme (%)': 100 * np.expm1(effet),
        'IC 95 % bas (%)': 100 * np.expm1(intervalle[0]),
        'IC 95 % haut (%)': 100 * np.expm1(intervalle[1]),
        'p-value': float(np.asarray(test_ecart.pvalue).reshape(-1)[0]),
    })
table_ecarts_csp = pd.DataFrame(ecarts_csp).set_index('CSP')
table_ecarts_csp['Significatif à 5 %'] = np.where(
    table_ecarts_csp['p-value'] < 0.05, 'Oui', 'Non'
)
table_ecarts_csp_affichage = table_ecarts_csp.copy()
table_ecarts_csp_affichage['p-value'] = (
    table_ecarts_csp_affichage['p-value'].map(formater_p_value)
)
display(table_ecarts_csp_affichage.round(2))
print(
    f"Observations : {int(modele_csp.nobs)} ; années : "
    f"{echantillon_csp['annee'].nunique()} ; R² : {modele_csp.rsquared:.3f}."
)
test_interactions_csp = test_conjoint(modele_csp, termes_interactions_csp)
print(
    'Test conjoint des interactions Femme × CSP : '
    f"F = {float(np.asarray(test_interactions_csp.statistic).reshape(-1)[0]):.2f}, "
    f"p {formater_p_value(float(np.asarray(test_interactions_csp.pvalue).reshape(-1)[0]))}."
)

echantillon_csp_tendance = echantillon_csp.copy()
echantillon_csp_tendance['trend'] = echantillon_csp_tendance['annee'] - 1996
categories_csp_non_reference = sorted(
    categorie for categorie in echantillon_csp_tendance['categorie_socio_prof'].unique()
    if categorie != 'Ouvriers'
)
colonnes_tendance_csp = {}
for indice, categorie in enumerate(categories_csp_non_reference):
    colonne = f'trend_csp_{indice}'
    echantillon_csp_tendance[colonne] = (
        echantillon_csp_tendance['categorie_socio_prof'].eq(categorie).astype(int)
        * echantillon_csp_tendance['trend']
    )
    colonnes_tendance_csp[categorie] = colonne

termes_triples_csp = [
    f'femme:{colonne}' for colonne in colonnes_tendance_csp.values()
]
formule_csp_tendance = (
    "np.log(valeur) ~ femme * "
    "C(categorie_socio_prof, Treatment(reference='Ouvriers')) "
    "+ femme:trend + C(annee)"
    + ''.join(
        f' + {colonne} + femme:{colonne}'
        for colonne in colonnes_tendance_csp.values()
    )
)
modele_csp_tendance = smf.ols(
    formula=formule_csp_tendance, data=echantillon_csp_tendance
).fit(
    cov_type='cluster',
    cov_kwds={'groups': echantillon_csp_tendance['annee'], 'use_correction': True},
)
if modele_csp_tendance.model.rank < len(modele_csp_tendance.params):
    raise ValueError('Le modèle avec interaction triple n’est pas de rang complet.')

if any(terme not in modele_csp_tendance.params.index for terme in termes_triples_csp):
    raise ValueError('Les interactions triples Femme × CSP × Trend attendues sont absentes.')

pentes_ecart_csp = []
for categorie in ['Ouvriers', *sorted(
    c for c in echantillon_csp_tendance['categorie_socio_prof'].unique()
    if c != 'Ouvriers'
)]:
    contraste = np.zeros(len(modele_csp_tendance.params))
    contraste[modele_csp_tendance.params.index.get_loc('femme:trend')] = 1
    if categorie != 'Ouvriers':
        terme = f'femme:{colonnes_tendance_csp[categorie]}'
        contraste[modele_csp_tendance.params.index.get_loc(terme)] = 1
    test_pente = modele_csp_tendance.t_test(contraste)
    pente = float(np.asarray(test_pente.effect).reshape(-1)[0])
    intervalle = np.asarray(test_pente.conf_int()).reshape(-1)
    pentes_ecart_csp.append({
        'CSP': categorie,
        'Variation annuelle de l’écart (%)': 100 * np.expm1(pente),
        'IC 95 % bas (%)': 100 * np.expm1(intervalle[0]),
        'IC 95 % haut (%)': 100 * np.expm1(intervalle[1]),
        'p-value': float(np.asarray(test_pente.pvalue).reshape(-1)[0]),
    })
table_pentes_ecart_csp = pd.DataFrame(pentes_ecart_csp).set_index('CSP')
table_pentes_ecart_csp_affichage = table_pentes_ecart_csp.copy()
table_pentes_ecart_csp_affichage['p-value'] = (
    table_pentes_ecart_csp_affichage['p-value'].map(formater_p_value)
)
display(table_pentes_ecart_csp_affichage.round(3))
test_triples_csp = test_conjoint(modele_csp_tendance, termes_triples_csp)
print(
    'Test conjoint des interactions Femme × CSP × Trend '
    '(variation des tendances par rapport aux ouvriers) : '
    f"F = {float(np.asarray(test_triples_csp.statistic).reshape(-1)[0]):.2f}, "
    f"p {formater_p_value(float(np.asarray(test_triples_csp.pvalue).reshape(-1)[0]))}."
)

# Écarts estimés par CSP avec leurs intervalles de confiance à 95 %.
fig, ax = plt.subplots(figsize=(9, 5))
positions = np.arange(len(table_ecarts_csp))
valeurs = table_ecarts_csp['Écart Femme–Homme (%)'].to_numpy()
borne_basse = table_ecarts_csp['IC 95 % bas (%)'].to_numpy()
borne_haute = table_ecarts_csp['IC 95 % haut (%)'].to_numpy()
ax.errorbar(
    valeurs, positions,
    xerr=np.vstack([valeurs - borne_basse, borne_haute - valeurs]),
    fmt='o', capsize=4, color='steelblue', ecolor='gray',
)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_yticks(positions, table_ecarts_csp.index)
ax.invert_yaxis()
ax.set(
    xlabel='Écart femmes-hommes sur le salaire moyen (%)',
    title='Écart salarial estimé selon la CSP (IC à 95 %)',
)
ax.grid(axis='x', alpha=0.25)
fig.tight_layout()
plt.show()


## 11. Régression n° 3 – Les écarts selon la position dans la distribution

Les données du projet contiennent les salaires moyens mensuels par sexe à sept positions de la distribution (D1, Q1, médiane, Q3, D9, P95 et P99). Elles permettent donc d’étudier si l’écart femmes-hommes est différent en bas et en haut de la distribution, sans utiliser le tableau EQTP08 lui-même.

Le modèle est :

$$\ln(Salaire_{s,q,t}) = \alpha + \beta_1 Femme_s + \beta_2 Quantile_q + \beta_3(Femme_s \times Quantile_q) + \lambda_t + \varepsilon_{s,q,t}$$

La médiane sert de référence. Les interactions indiquent comment l’écart de rémunération à chaque quantile diffère de l’écart à la médiane. Les erreurs-types sont regroupées par année ; les salaires sont des statistiques agrégées, et non des observations individuelles.

In [ ]:
quantiles_ordonnes = [
    '10e centile (D1 - 10% gagnent moins)',
    '1er quartile (Q1 - 25% gagnent moins)',
    'Médiane (50% gagnent moins/plus)',
    '3e quartile (Q3 - 75% gagnent moins)',
    '90e centile (D9 - 10% gagnent plus)',
    '95e centile (Top 5%)',
    '99e centile (Top 1%)',
]
reference_quantile = 'Médiane (50% gagnent moins/plus)'
echantillon_quantiles = prive.loc[
    prive['indicateur'].eq(SALAIRE)
    & prive['centile_salaire'].isin(quantiles_ordonnes)
    & prive['geo'].eq('F')
    & prive['secteur_activite'].eq('Ensemble des secteurs')
    & prive['tranche_age'].eq('Ensemble')
    & prive['taille_entreprise'].eq('Ensemble')
    & prive['temps_travail'].eq('Ensemble')
    & prive['categorie_socio_prof'].eq('Ensemble')
    & prive['sexe'].isin(['Femme', 'Homme']),
    ['annee', 'sexe', 'centile_salaire', 'valeur'],
].copy()
echantillon_quantiles['annee'] = pd.to_numeric(
    echantillon_quantiles['annee'], errors='coerce'
)
echantillon_quantiles['valeur'] = pd.to_numeric(
    echantillon_quantiles['valeur'], errors='coerce'
)
if echantillon_quantiles[['annee', 'valeur']].isna().any().any():
    raise ValueError('Des années ou des salaires manquent dans les données par quantile.')
echantillon_quantiles['annee'] = echantillon_quantiles['annee'].astype(int)
if (echantillon_quantiles['valeur'] <= 0).any():
    raise ValueError('Le logarithme du salaire exige des valeurs strictement positives.')

cles_quantiles = ['annee', 'sexe', 'centile_salaire']
if echantillon_quantiles.duplicated(cles_quantiles).any():
    raise ValueError(f'Plusieurs observations pour les mêmes clés : {cles_quantiles}')
annees_quantiles = sorted(echantillon_quantiles['annee'].unique())
attendues_quantiles = pd.MultiIndex.from_product(
    [annees_quantiles, ['Femme', 'Homme'], quantiles_ordonnes],
    names=cles_quantiles,
)
observees_quantiles = pd.MultiIndex.from_frame(echantillon_quantiles[cles_quantiles])
manquantes_quantiles = attendues_quantiles.difference(observees_quantiles)
if len(manquantes_quantiles):
    raise ValueError(
        f"Panel sexe-quantile incomplet : {len(manquantes_quantiles)} combinaisons manquent."
    )

formule_quantiles = (
    "np.log(valeur) ~ femme * "
    f"C(centile_salaire, Treatment(reference='{reference_quantile}')) + C(annee)"
)
echantillon_quantiles['femme'] = (
    echantillon_quantiles['sexe'].eq('Femme').astype(int)
)
modele_quantiles = smf.ols(
    formula=formule_quantiles, data=echantillon_quantiles
).fit(
    cov_type='cluster',
    cov_kwds={'groups': echantillon_quantiles['annee'], 'use_correction': True},
)
if modele_quantiles.model.rank < len(modele_quantiles.params):
    raise ValueError('Le modèle sexe-quantile n’est pas de rang complet.')

ecarts_quantiles = []
for quantile in quantiles_ordonnes:
    contraste = np.zeros(len(modele_quantiles.params))
    contraste[modele_quantiles.params.index.get_loc('femme')] = 1
    if quantile != reference_quantile:
        termes_quantile = [
            terme for terme in modele_quantiles.params.index
            if terme.startswith('femme:')
            and 'C(centile_salaire' in terme
            and terme.endswith(f'[T.{quantile}]')
        ]
        if len(termes_quantile) != 1:
            raise KeyError(f'Interaction sexe-quantile introuvable ou ambiguë : {quantile}')
        contraste[modele_quantiles.params.index.get_loc(termes_quantile[0])] = 1
    test_ecart = modele_quantiles.t_test(contraste)
    effet = float(np.asarray(test_ecart.effect).reshape(-1)[0])
    intervalle = np.asarray(test_ecart.conf_int()).reshape(-1)
    ecarts_quantiles.append({
        'Quantile': quantile,
        'Écart Femme–Homme (%)': 100 * np.expm1(effet),
        'IC 95 % bas (%)': 100 * np.expm1(intervalle[0]),
        'IC 95 % haut (%)': 100 * np.expm1(intervalle[1]),
        'p-value': float(np.asarray(test_ecart.pvalue).reshape(-1)[0]),
    })
table_ecarts_quantiles = pd.DataFrame(ecarts_quantiles).set_index('Quantile')
table_ecarts_quantiles['Significatif à 5 %'] = np.where(
    table_ecarts_quantiles['p-value'] < 0.05, 'Oui', 'Non'
)
table_ecarts_quantiles_affichage = table_ecarts_quantiles.copy()
table_ecarts_quantiles_affichage['p-value'] = (
    table_ecarts_quantiles_affichage['p-value'].map(formater_p_value)
)
display(table_ecarts_quantiles_affichage.round(2))

termes_interactions_quantiles = [
    terme for terme in modele_quantiles.params.index
    if terme.startswith('femme:') and 'C(centile_salaire' in terme
]
if len(termes_interactions_quantiles) != len(quantiles_ordonnes) - 1:
    raise ValueError('Les interactions sexe-quantile attendues ne sont pas toutes estimées.')
test_interactions_quantiles = test_conjoint(
    modele_quantiles, termes_interactions_quantiles
)
print(
    f"Observations : {int(modele_quantiles.nobs)} ; années : "
    f"{echantillon_quantiles['annee'].nunique()} ; R² : {modele_quantiles.rsquared:.3f}."
)
print(
    'Test conjoint des interactions Femme × Quantile '
    '(H₀ : l’écart femmes-hommes est identique à tous les quantiles) : '
    f"F = {float(np.asarray(test_interactions_quantiles.statistic).reshape(-1)[0]):.2f}, "
    f"p {formater_p_value(float(np.asarray(test_interactions_quantiles.pvalue).reshape(-1)[0]))}."
)

# Écarts estimés par quantile avec leurs intervalles de confiance à 95 %.
fig, ax = plt.subplots(figsize=(10, 5))
positions = np.arange(len(table_ecarts_quantiles))
valeurs = table_ecarts_quantiles['Écart Femme–Homme (%)'].to_numpy()
borne_basse = table_ecarts_quantiles['IC 95 % bas (%)'].to_numpy()
borne_haute = table_ecarts_quantiles['IC 95 % haut (%)'].to_numpy()
ax.errorbar(
    positions, valeurs,
    yerr=np.vstack([valeurs - borne_basse, borne_haute - valeurs]),
    fmt='o', capsize=4, color='darkorange', ecolor='gray',
)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(positions, ['D1', 'Q1', 'Médiane', 'Q3', 'D9', 'P95', 'P99'])
ax.set(
    xlabel='Position dans la distribution des salaires',
    ylabel='Écart femmes-hommes estimé (%)',
    title='Écart salarial estimé selon le quantile (IC à 95 %)',
)
ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
plt.show()


## 12. Conclusion d’ensemble

Les trois régressions examinent l’écart femmes-hommes sous des angles complémentaires : son évolution temporelle, ses différences selon la CSP et sa variation le long de la distribution des salaires. Les erreurs-types tiennent compte de la dépendance temporelle dans la première analyse (Newey–West) et regroupent les observations par année dans les modèles par CSP et quantile.

Les données du dépôt portent sur le salaire net moyen mensuel en euros constants ; elles ne sont pas les tableaux EQTP02 et EQTP08 et ne donnent pas directement le salaire annuel moyen EQTP. Les résultats restent des associations calculées sur des moyennes ou quantiles agrégés. Ils ne permettent ni d’isoler des trajectoires individuelles comparables, ni d’identifier l’effet causal d’une politique d’égalité ou une discrimination individuelle.